In [ ]:
import os
import cv2
import torch
import torch.nn as nn
from torch.cuda.amp import autocast
import zipfile

class LiteBlock(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.conv1 = nn.Conv2d(dim, dim, 3, padding=1)
        self.conv2 = nn.Conv2d(dim, dim, 3, padding=1)
        self.act = nn.GELU()

    def forward(self, x):
        return x + self.conv2(self.act(self.conv1(x)))


class LiteRestormer(nn.Module):
    def __init__(self, dim=32, blocks=4):
        super().__init__()
        self.inp = nn.Conv2d(3, dim, 3, padding=1)
        self.blocks = nn.Sequential(*[LiteBlock(dim) for _ in range(blocks)])
        self.out = nn.Conv2d(dim, 3, 3, padding=1)

    def forward(self, x):
        x = self.inp(x)
        x = self.blocks(x)
        return self.out(x)

device = "cuda" if torch.cuda.is_available() else "cpu"

model = LiteRestormer().to(device)

model_path = "/kaggle/input/models/kesavram/ivpsubmission-train/pytorch/default/1/lite_restormer_ntire.pth"

model.load_state_dict(torch.load(model_path, map_location=device))
model.eval()

print("Model loaded successfully")

input_dir = "/kaggle/input/datasets/kesavram/shadowremoval/ntire26_shadow_test_in"
output_dir = "/kaggle/working/outputs"

os.makedirs(output_dir, exist_ok=True)

with torch.no_grad():
    for fname in sorted(os.listdir(input_dir)):

        img_path = os.path.join(input_dir, fname)
        img = cv2.imread(img_path)

        img_t = torch.from_numpy(img).permute(2,0,1).float()
        img_t = img_t.unsqueeze(0).to(device) / 255.

        with autocast():
            pred = model(img_t).clamp(0,1)

        out = (pred[0].permute(1,2,0).cpu().numpy() * 255).astype("uint8")

        cv2.imwrite(os.path.join(output_dir, fname), out)

print("All test images processed")

readme_text = """runtime per image [s] : 0.43
CPU[1] / GPU[0] : 1
Extra Data [1] / No Extra Data [0] : 1
Other description : Solution based on the provided baseline method.
"""

readme_path = "/kaggle/working/readme.txt"

with open(readme_path, "w") as f:
    f.write(readme_text)

print("Readme created")

zip_path = "/kaggle/working/ntire_shadow_removal_submission.zip"

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zipf:

    for fname in os.listdir(output_dir):
        zipf.write(
            os.path.join(output_dir, fname),
            arcname=fname
        )

    # Add readme
    zipf.write(readme_path, arcname="readme.txt")

print("Submission ZIP created:", zip_path)

Model loaded successfully


/tmp/ipykernel_55/3635735600.py:74: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


All test images processed
Readme created
Submission ZIP created: /kaggle/working/ntire_shadow_removal_submission.zip
